# 04 - Model Training & Ablation Study

Train the hybrid DC-LLM prediction model and run ablation studies.

**Key experiment:** Demonstrate that combining DC + LLM features
outperforms either feature set alone.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data_collection.stock_fetcher import StockDataFetcher
from src.data_collection.conflict_tracker import ConflictEventTracker
from src.models.feature_engineering import FeatureEngineer
from src.models.hybrid_model import HybridDCLLMPredictor
from src.models.baselines import BaselineModels
from src.visualization.dc_plots import DCVisualizer
from src.visualization.heatmaps import HeatmapVisualizer

## 1. Build Feature Matrix

In [ ]:
fetcher = StockDataFetcher()
sp500 = fetcher.fetch_symbol('^GSPC', '2015-01-01', '2025-12-31')
prices = sp500['close']
volume = sp500.get('volume')

events_tracker = ConflictEventTracker()
events = events_tracker.get_combined_geopolitical_events()

# Build features
engineer = FeatureEngineer(dc_threshold=0.02)
features = engineer.build_unified_features(
    prices, volume=volume, events_df=events
)
labels = engineer.build_labels(prices, horizon=5, method='direction')

print(f'Features: {features.shape}')
print(f'Labels distribution: {labels.value_counts().to_dict()}')
features.head()

## 2. Baselines

In [ ]:
baselines = BaselineModels()
baseline_results = baselines.run_all_baselines(prices, labels)
baseline_results

## 3. Ablation Study

In [ ]:
model = HybridDCLLMPredictor()
ablation = model.run_ablation_study(features, labels, model_name='xgboost')

viz = DCVisualizer()
fig = viz.plot_ablation_results(ablation)
fig.show()

ablation

## 4. Model Comparison

In [ ]:
comparison = model.model_comparison(features, labels)
comparison

## 5. Feature Importance

In [ ]:
result = model.train_and_evaluate(features, labels, model_name='xgboost')

if 'top_features' in result:
    heatmap_viz = HeatmapVisualizer()
    fig = heatmap_viz.plot_feature_importance(result['top_features'], top_n=20)
    fig.show()

    print('\nTop 20 Features:')
    print(result['top_features'])

## 6. Final Predictions

In [ ]:
predictions = model.get_final_predictions(features, labels, model_name='xgboost')

print(f'Test Accuracy: {predictions["correct"].mean():.4f}')
predictions.tail(20)